# MINE4201 - SR - Taller 1 - Experimentación y optimización para K vecinos más cercanos

**Grupo 3**

Lina María Ojeda Amaya Código: 202112324

Diego Felipe Tolosa Código: 201413235

Juan Manuel Rivera López Código: 201534131

Johana Alejandra Rátiva Mora Código: 202513844

# Importación de librerías y datos

In [1]:
import numpy as np
import os
import pandas as pd
import random
from matplotlib import pyplot as plt
import time

from scipy.sparse import csr_matrix

from sklearn.preprocessing import normalize
from tqdm import tqdm

#Para garantizar reproducibilidad en resultados
seed = 10
random.seed(seed)
np.random.seed(seed)

In [2]:
base_path = "C:/Users/jmriv/OneDrive - Universidad de los andes/Semestres uniandes/2026-1/Sistemas de recomendación/Talleres"
links_path = os.path.join(base_path, "link.csv")
movies_path = os.path.join(base_path, "movie.csv")
ratings_path = os.path.join(base_path, "rating.csv")

In [3]:
ratings = pd.read_csv(ratings_path)
ratings.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [4]:
movies = pd.read_csv(movies_path)
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
test = ratings.groupby("userId", group_keys=False).sample(frac=0.2, random_state=42)
train = ratings.drop(test.index)

In [5]:
user_cat = train.userId.astype("category")
item_cat = train.movieId.astype("category")

user_ids = user_cat.cat.codes
item_ids = item_cat.cat.codes

R_train = csr_matrix(
    (train.rating, (user_ids, item_ids))
)

In [6]:
user_map = dict(zip(user_cat.cat.categories, range(len(user_cat.cat.categories))))
item_map = dict(zip(item_cat.cat.categories, range(len(item_cat.cat.categories))))

# Evaluación de similitud usuario-usuario

In [20]:
N = R_train.shape[0]

counts = np.diff(R_train.indptr)
sums = np.add.reduceat(R_train.data, R_train.indptr[:-1])
means = sums / counts

In [ ]:
def predict_rating(
        R,
        userId,
        movieId,
        neighbours,
        sims,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50,):

    # --- get ratings for the movie ---
    ratings = R[neighbours, movieId].toarray().flatten()

    # keep neighbours that rated the movie
    mask = (ratings > 0) & (sims >= sim_threshold)

    neighbours = neighbours[mask]
    sims = sims[mask]
    ratings = ratings[mask]

    if len(ratings) == 0:
        return np.nan

    # --- mean centered ratings ---
    centered = ratings - means[neighbours]

    # --- McLaughlin significance weighting ---
    if weighting:

        intersections = (B[userId] @ B[neighbours].T).toarray().flatten()

        weights = np.minimum(intersections / beta, 1.0)

        sims *= weights

    numerator = np.sum(sims * centered)
    denominator = np.sum(np.abs(sims))

    if denominator == 0:
        return np.nan

    pred = means[userId] + numerator / denominator

    return pred

In [22]:
def evaluate_rmse(
        R,
        test_df,
        neighbors_list,
        sims_list,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None):

    preds = []
    truths = []

    users = test_df.user_idx.values
    movies = test_df.movie_idx.values
    ratings = test_df.rating.values

    for i in range(len(users)):

        user_idx = users[i]
        movie_idx = movies[i]
        rating = ratings[i]

        pred = predict_rating(
            R,
            user_idx,
            movie_idx,
            neighbors_list[user_idx],
            sims_list[user_idx],
            K,
            sim_threshold,
            means,
            weighting,
            B
        )

        if not np.isnan(pred):
            preds.append(pred)
            truths.append(rating)

    preds = np.array(preds)
    truths = np.array(truths)

    rmse = np.sqrt(np.mean((preds - truths) ** 2))

    return rmse

## Sub muestra de test (n=5.000)

In [33]:
results = []

B = R_train.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(5_000, random_state=42)

for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.3,0.5], desc="sim_threshold", leave=False):
            for weight in [True, False]:
                rmse = evaluate_rmse(
                    R_train,
                    test_filtered,
                    neighbours,
                    sims,
                    K,
                    sim_threshold,
                    means,
                    weight,
                    B
                )

                results.append((metric, K, sim_threshold, weight, rmse))
                print((metric, K, sim_threshold, weight, rmse))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.05408
max sim: 0.9224


('cosine', 25, 0, True, 0.8769094657817021)


('cosine', 25, 0, False, 0.8754917505297362)
('cosine', 25, 0.3, True, 0.8733900170970228)


('cosine', 25, 0.3, False, 0.8720686704904383)
('cosine', 25, 0.5, True, 0.8984034202681065)


...

('jaccard', 100, 0, True, 0.8541862557110242)


('jaccard', 100, 0, False, 0.8537566672461587)
('jaccard', 100, 0.3, True, 0.8851231756707167)


('jaccard', 100, 0.3, False, 0.8855272251516845)
('jaccard', 100, 0.5, True, 0.9192466520257577)




Similarity metric: 100%|██████████| 3/3 [07:57<00:00, 159.14s/it]

('jaccard', 100, 0.5, False, 0.9213708555556887)


In [34]:
results

[('cosine', 25, 0, True, 0.8769094657817021),
 ('cosine', 25, 0, False, 0.8754917505297362),
 ('cosine', 25, 0.3, True, 0.8733900170970228),
 ('cosine', 25, 0.3, False, 0.8720686704904383),
 ('cosine', 25, 0.5, True, 0.8984034202681065),
 ('cosine', 25, 0.5, False, 0.8994676945490273),
 ('cosine', 50, 0, True, 0.8537564652463303),
 ('cosine', 50, 0, False, 0.8523012503228641),
 ('cosine', 50, 0.3, True, 0.8509605651302702),
 ('cosine', 50, 0.3, False, 0.8491847943040124),
 ('cosine', 50, 0.5, True, 0.8825747326694313),
 ('cosine', 50, 0.5, False, 0.8839743801272706),
 ('cosine', 100, 0, True, 0.8464318607620699),
 ('cosine', 100, 0, False, 0.8449797573858153),
 ('cosine', 100, 0.3, True, 0.8467263808674127),
 ('cosine', 100, 0.3, False, 0.8452799129614096),
 ('cosine', 100, 0.5, True, 0.8781622591077123),
 ('cosine', 100, 0.5, False, 0.8797395197854984),
 ('pearson', 25, 0, True, 0.8734886457752691),
 ('pearson', 25, 0, False, 0.8691050174067901),
 ('pearson', 25, 0.3, True, 0.84532472

Viendo que el coeficiente de Jaccard tiene los peores resultados en la exploración incial, se excluirá en iteraciones futuras. Además, viendo que ponderar la similutudes usando el método de McLaughlin tiene un efecto moderado, y que el calculo supone una multiplicación matricial adicional, se excluirá por el momento.

## Submuestra de test (n=250.000)

In [35]:
results = []

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

test_filtered = test_filtered.sample(250_000, random_state=42)

for metric in tqdm(["cosine", "pearson"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/users_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.3,0.5], desc="sim_threshold", leave=False):

            rmse = evaluate_rmse(
                R_train,
                test_filtered,
                neighbours,
                sims,
                K,
                sim_threshold,
                means,
                weighting=False,
                B=None
            )

            results.append((metric, K, sim_threshold, weight, rmse))
            print((metric, K, sim_threshold, weight, rmse))

Similarity metric:   0%|          | 0/2 [00:00<?, ?it/s]

min sim: 0.05408
max sim: 0.9224


('cosine', 25, 0, False, 0.8806656033893725)


('cosine', 25, 0.3, False, 0.8741087211940434)


('cosine', 25, 0.5, False, 0.8753475981812547)


...

('pearson', 50, 0.5, False, 0.8177949505761316)


('pearson', 100, 0, False, 0.8581761219436683)


('pearson', 100, 0.3, False, 0.8513725760154347)




Similarity metric: 100%|██████████| 2/2 [1:50:15<00:00, 3307.85s/it]  

('pearson', 100, 0.5, False, 0.8170917909421229)


In [36]:
results

[('cosine', 25, 0, False, 0.8806656033893725),
 ('cosine', 25, 0.3, False, 0.8741087211940434),
 ('cosine', 25, 0.5, False, 0.8753475981812547),
 ('cosine', 50, 0, False, 0.8626560533342876),
 ('cosine', 50, 0.3, False, 0.8573224019235202),
 ('cosine', 50, 0.5, False, 0.8684518666093048),
 ('cosine', 100, 0, False, 0.8511609839977688),
 ('cosine', 100, 0.3, False, 0.8477897557746596),
 ('cosine', 100, 0.5, False, 0.864028738041872),
 ('pearson', 25, 0, False, 0.8892202755242916),
 ('pearson', 25, 0.3, False, 0.8634113679362047),
 ('pearson', 25, 0.5, False, 0.817639193129947),
 ('pearson', 50, 0, False, 0.8733750963456223),
 ('pearson', 50, 0.3, False, 0.8562543751332603),
 ('pearson', 50, 0.5, False, 0.8177949505761316),
 ('pearson', 100, 0, False, 0.8581761219436683),
 ('pearson', 100, 0.3, False, 0.8513725760154347),
 ('pearson', 100, 0.5, False, 0.8170917909421229)]

# Evaluación de similitud item-item

In [141]:
R_items = R_train.T

N = R_items.shape[0]

counts = np.diff(R_items.indptr)
sums = np.add.reduceat(R_items.data, R_items.indptr[:-1])
item_means = np.divide(sums, counts, out=np.zeros_like(sums), where=counts!=0)

In [149]:
def predict_rating(
        R,
        userId,
        movieId,
        neighbours,
        sims,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None,
        beta=50,):

    # --- get ratings for the movie ---
    ratings = R[userId, neighbours].toarray().flatten()

    # keep neighbours that rated the movie
    mask = (ratings > 0) & (sims >= sim_threshold)

    neighbours = neighbours[mask]
    sims = sims[mask]
    ratings = ratings[mask]

    if len(ratings) == 0:
        return np.nan

    # --- mean centered ratings ---
    centered = ratings - means[userId]

    # --- McLaughlin significance weighting ---
    if weighting:

        intersections = (B[movieId] @ B[neighbours].T).toarray().flatten()

        weights = np.minimum(intersections / beta, 1.0)

        sims *= weights

    numerator = np.sum(sims * centered)
    denominator = np.sum(np.abs(sims))

    if denominator == 0:
        return np.nan

    pred = means[movieId] + numerator / denominator

    return pred

In [152]:
def evaluate_rmse(
        R,
        test_df,
        neighbors_list,
        sims_list,
        K,
        sim_threshold,
        means,
        weighting=False,
        B=None):

    preds = []
    truths = []

    users = test_df.user_idx.values
    movies = test_df.movie_idx.values
    ratings = test_df.rating.values

    for i in range(len(users)):

        user_idx = users[i]
        movie_idx = movies[i]
        rating = ratings[i]

        pred = predict_rating(
            R,
            user_idx,
            movie_idx,
            neighbors_list[movie_idx],
            sims_list[movie_idx],
            K,
            sim_threshold,
            means,
            weighting,
            B
        )

        if not np.isnan(pred):
            preds.append(pred)
            truths.append(rating)
        
        if i%100 == 0:
            print(i)

    preds = np.array(preds)
    truths = np.array(truths)

    rmse = np.sqrt(np.mean((preds - truths) ** 2))

    return rmse

## Submuestra de test (n=5.000)

In [144]:
results = []

B = R_items.copy()
B.data = np.ones_like(B.data)

test_filtered = test[
    test.userId.isin(user_map) &
    test.movieId.isin(item_map)
].copy()

test_filtered["user_idx"] = test_filtered.userId.map(user_map)
test_filtered["movie_idx"] = test_filtered.movieId.map(item_map)

# 5.000 se calculan en 15 minutos
test_filtered = test_filtered.sample(5_000, random_state=42)

In [150]:
metric="cosine"
with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
    top_k_indices = data["indices"]
    top_k_sims = data["sims"]
print("min sim:", top_k_sims.min())
print("max sim:", top_k_sims.max())

K =25

neighbours = top_k_indices[:, :K]
sims = top_k_sims[:, :K]

sim_threshold = 0.3
weight = True

min sim: 0.0
max sim: 1.0


In [153]:
rmse = evaluate_rmse(
            R_train,
            test_filtered,
            neighbours,
            sims,
            K,
            sim_threshold,
            means,
            weight,
            B
        )

print((metric, K, sim_threshold, weight, rmse))

0
100
200
300
400
500
600
700
800
900
1000
1100


KeyboardInterrupt: 

In [ ]:
for metric in tqdm(["cosine", "pearson", "jaccard"], desc="Similarity metric"):
    with np.load(f"Train_neighbours/items_{metric}_top100_neighbors.npz") as data:
        top_k_indices = data["indices"]
        top_k_sims = data["sims"]
    print("min sim:", top_k_sims.min())
    print("max sim:", top_k_sims.max())

    for K in tqdm([25, 50, 100], desc="K neighbors", leave=False):

        neighbours = top_k_indices[:, :K]
        sims = top_k_sims[:, :K]

        for sim_threshold in tqdm( [0,0.3,0.5], desc="sim_threshold", leave=False):
            for weight in [True, False]:
                rmse = evaluate_rmse(
                    R_train,
                    test_filtered,
                    neighbours,
                    sims,
                    K,
                    sim_threshold,
                    means,
                    weight,
                    B
                )

                results.append((metric, K, sim_threshold, weight, rmse))
                print((metric, K, sim_threshold, weight, rmse))

Similarity metric:   0%|          | 0/3 [00:00<?, ?it/s]

min sim: 0.0
max sim: 1.0


('cosine', 25, 0, True, 1.005884672292797)


('cosine', 25, 0, False, 1.0056557408107982)
('cosine', 25, 0.3, True, 1.018338013299828)


('cosine', 25, 0.3, False, 1.0183418489961007)
('cosine', 25, 0.5, True, 1.0282198152268736)


...

('pearson', 25, 0, False, 1.0053548899690898)
('pearson', 25, 0.3, True, 1.0186657317695755)


('pearson', 25, 0.3, False, 1.0186697307054762)
('pearson', 25, 0.5, True, 1.0282214619919625)


('pearson', 25, 0.5, False, 1.0282214619871746)




Similarity metric:  33%|███▎      | 1/3 [1:34:39<3:09:18, 5679.33s/it]


KeyboardInterrupt: 